# Memahami Jaringan Saraf Rekuren: SimpleRNN dan LSTM

**Buku:** *Deep Learning dengan Python* — Bab 6, Deep Learning untuk Teks dan Sekuens

Implementasi RNN dengan NumPy, lapisan rekuren Keras, lalu klasifikasi sentimen IMDB dengan SimpleRNN dan LSTM.

> Diadaptasi dari notebook pendamping *Deep Learning with Python* (F. Chollet, Manning; kode berlisensi MIT) dan diperbarui ke **Keras 3** untuk buku *Deep Learning dengan Python: Konsep, Kode, dan Penerapan Modern* (B. S. Negara).

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+, Keras 3. Kode bersifat *backend-agnostik*: dapat dijalankan dengan backend TensorFlow, JAX, atau PyTorch.

**Perubahan utama dari versi asli (Keras 2.0.8):**
- `keras.preprocessing.sequence.pad_sequences` → `keras.utils.pad_sequences`.
- Ditambahkan implementasi RNN NumPy (ada di buku, tidak ada di notebook asli).
- Model memakai `keras.Input` agar `summary()` menampilkan bentuk lengkap.

In [ ]:
import os
# Pilih backend: "tensorflow", "jax", atau "torch"
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())

## 1. RNN dengan NumPy
RNN hanyalah perulangan `for` yang membawa *keadaan* dari langkah sebelumnya.

In [ ]:
import numpy as np

langkah_waktu, fitur_masuk, fitur_keluar = 100, 32, 64
rng = np.random.default_rng(0)
masukan = rng.random((langkah_waktu, fitur_masuk))
keadaan = np.zeros((fitur_keluar,))
W = rng.random((fitur_keluar, fitur_masuk))
U = rng.random((fitur_keluar, fitur_keluar))
b = rng.random((fitur_keluar,))

keluaran = []
for x_t in masukan:
    y_t = np.tanh(W @ x_t + U @ keadaan + b)
    keluaran.append(y_t)
    keadaan = y_t
keluaran = np.stack(keluaran)
print(keluaran.shape)        # (100, 64)

## 2. Lapisan `SimpleRNN`: keluaran terakhir vs seluruh sekuens

In [ ]:
from keras import layers

for urutan_penuh in [False, True]:
    m = keras.Sequential([keras.Input(shape=(None,), dtype="int64"),
                          layers.Embedding(10000, 32),
                          layers.SimpleRNN(32, return_sequences=urutan_penuh)])
    print("return_sequences =", urutan_penuh, "->", m.output_shape)

bertumpuk = keras.Sequential([keras.Input(shape=(None,), dtype="int64"),
                              layers.Embedding(10000, 32),
                              layers.SimpleRNN(32, return_sequences=True),
                              layers.SimpleRNN(32, return_sequences=True),
                              layers.SimpleRNN(32)])
bertumpuk.summary()

## 3. IMDB dengan SimpleRNN

In [ ]:
from keras.datasets import imdb

MAKS_KATA = 10000     # kosakata: 10.000 kata paling sering
PANJANG = 500         # potong ulasan setelah 500 kata
(x_latih, y_latih), (x_uji, y_uji) = imdb.load_data(num_words=MAKS_KATA)
x_latih = keras.utils.pad_sequences(x_latih, maxlen=PANJANG)
x_uji = keras.utils.pad_sequences(x_uji, maxlen=PANJANG)
print(x_latih.shape, x_uji.shape)

In [ ]:
import matplotlib.pyplot as plt

def plot_riwayat(riwayat, metrik="accuracy", judul=""):
    h = riwayat.history
    epoch = range(1, len(h["loss"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(epoch, h[metrik], "o", ms=3, label="Latih")
    ax[0].plot(epoch, h["val_" + metrik], "-", label="Validasi")
    ax[0].set(xlabel="Epoch", ylabel=metrik, title=judul)
    ax[1].plot(epoch, h["loss"], "o", ms=3, label="Latih")
    ax[1].plot(epoch, h["val_loss"], "-", label="Validasi")
    ax[1].set(xlabel="Epoch", ylabel="Loss", title=judul)
    for a in ax: a.legend()
    plt.tight_layout(); plt.show()

model = keras.Sequential([keras.Input(shape=(None,), dtype="int64"),
                          layers.Embedding(MAKS_KATA, 32),
                          layers.SimpleRNN(32),
                          layers.Dense(1, activation="sigmoid")])
model.compile(optimizer="rmsprop", loss="binary_crossentropy", metrics=["accuracy"])
riwayat = model.fit(x_latih, y_latih, epochs=10, batch_size=128, validation_split=0.2)
plot_riwayat(riwayat, judul="SimpleRNN")

# Untuk verifikasi angka buku (Subbab 6.2)
print(f"SimpleRNN: val acc terbaik {max(riwayat.history['val_accuracy']):.2%}")

## 4. IMDB dengan LSTM
LSTM menambah "jalur bawa" (*carry*) yang mencegah sinyal lama memudar (*vanishing gradient*).

In [ ]:
model = keras.Sequential([keras.Input(shape=(None,), dtype="int64"),
                          layers.Embedding(MAKS_KATA, 32),
                          layers.LSTM(32),
                          layers.Dense(1, activation="sigmoid")])
model.compile(optimizer="rmsprop", loss="binary_crossentropy", metrics=["accuracy"])
riwayat = model.fit(x_latih, y_latih, epochs=10, batch_size=128, validation_split=0.2)
plot_riwayat(riwayat, judul="LSTM")
print(f"Akurasi uji LSTM: {model.evaluate(x_uji, y_uji, verbose=0)[1]:.2%}")